# Historical Synthetic Must: Calibration and Estimability

This notebook independently reproduces the calibration, Fisher information, eigen-direction and profile-likelihood workflow used in `fermentation_estimability_old_vs_lot1`, restricted to the historical `synthetic` medium.

## tl;dr

The numerical conclusions are generated below from a fresh medium-specific fit. The final report states which parameters are locally supported, which remain confounded, and how well the fitted model reproduces each observed state.

## Context & Methods

The core parameter block is re-estimated in log space using the same bounds, residual scaling and deterministic multistart as the reference notebook. The local Fisher information matrix is

$$
F(\theta) = J(\theta)^\mathsf{T} J(\theta),
$$

where $J$ is the Jacobian of scaled residuals with respect to log-parameters. Profile likelihood is used as a nonlinear check for `Kd0` and `qN`; the three-point grid is a screening profile, not a final confidence interval.

No continuous process-sensor files are added to the synthetic historical subset in this notebook.

### Key Assumptions

- Secondary and aroma parameters are evaluated conditionally at the shared prior values; only the core fermentation block is re-estimated here.
- CO2 flow and cumulative CO2 production are different observables. Sensor values remain diagnostic until a gas-liquid observation layer is calibrated.
- Missing observations stay missing. They are never replaced with zeros.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

from laboratory_2026 import run_estimability_historical_by_medium as analysis

MEDIUM = 'synthetic'
RESULTS = analysis.results_dir(MEDIUM)


## Run Analysis

In [ ]:
analysis.run_medium_analysis(
    MEDIUM,
    profile_parameters=("Kd0", "qN"),
    grid_points=3,
    fit_nfev=300,
    profile_nfev=45,
    step=0.04,
)

## Data

In [ ]:
batch_summary = pd.read_csv(RESULTS / "batch_summary.csv")
support = pd.read_csv(RESULTS / "measurement_support.csv")
display(batch_summary)
display(support)

## Results

In [ ]:
fit_candidates = pd.read_csv(RESULTS / "core_fit_candidate_summary.csv")
fit_summary = pd.read_csv(RESULTS / "core_fit_summary.csv")
fim_metrics = pd.read_csv(RESULTS / "fim_metrics.csv")
display(fit_candidates[["case", "seed", "final_wsse", "n_residuals", "nfev", "success"]])
display(fit_summary)
display(fim_metrics)

### Parameter Estimability

In [ ]:
estimability = pd.read_csv(RESULTS / "estimability_extended.csv")
display(estimability.sort_values("std_log_approx"))

In [ ]:
display(Image(filename=str(RESULTS / "figures" / "estimability_std_log.png")))

### Eigenvalue Diagnostics

In [ ]:
weak = pd.read_csv(RESULTS / "weak_directions_extended.csv")
display(weak)
display(Image(filename=str(RESULTS / "figures" / "extended_fim_eigenvalues.png")))

### Profile Likelihood

In [ ]:
profile_summary = pd.read_csv(RESULTS / "profile_likelihood_summary_core.csv")
profile = pd.read_csv(RESULTS / "profile_likelihood_core.csv")
display(profile_summary)
display(profile)

In [ ]:
display(Image(filename=str(RESULTS / "figures" / "profile_likelihood.png")))

### Curve-Fit Checks

In [ ]:
core_curve_metrics = pd.read_csv(RESULTS / "core_fit_metrics_by_batch_state.csv")
secondary_curve_metrics = pd.read_csv(RESULTS / "secondary_fit_metrics_by_batch_state.csv")
display(core_curve_metrics.groupby("state").agg(n=("n", "sum"), median_RMSE=("rmse", "median"), max_RMSE=("rmse", "max")))
display(secondary_curve_metrics.groupby("state").agg(n=("n", "sum"), median_RMSE=("rmse", "median"), max_RMSE=("rmse", "max")))

## Takeaways

In [ ]:
report = (RESULTS / "estimability_historical_synthetic_report.md").read_text(encoding="utf-8")
print(report)